In [89]:
# In this section I am importing all the libraries I will need
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

In [90]:
# In this section I am defining variables (arrays) that I would need to implement the method
# This is for a beam that is simply supported at both ends with uniformly distrubted load throughout beam span
# Physical Parameters of the beam

E = 210 * (10 ** 9)         # Young's Modulus, Pa
L = 10                      # Beam Length, m
q = 160.0                   # Uniform distributed load, N/m
b = 0.08                    # Width, m
d = 0.10                    # Depth, m

I = b * (d**3) / 12         # Second Moment of Area, m^4

N = 40                      # Number of Intervals
h = L / N                   # Spacing between Nodes

x = np.linspace(0,L,N+1)      # Nodal Coordinates
v = np.zeros(N+1)           # Deflection Array

v_inside = np.zeros(N-1)    # Unknown deflection values

A = np.zeros((N-1,N-1))     # Matrix with coefficients of deflection
b_vect = np.zeros(N-1)      # Values of Deflection, RHS of equation


In [91]:
# In this section I am setting the boundary conditions/initial values

v[0] = 0                    # Value of deflection at LHS Support
v[N] = 0                    # Value of defleciton at RHS Support

# Bending moment boundary conditions, v''(0) = 0 and v''(L) = 0 are used in the numerical methods part of the code, since this information is used to calculate v-1 and vn+1 
# which are points outside domain, conditions are incorporated into the first and last rows of coefficient matrix A below.


In [92]:
 # In this section I am implementing the numerical method

for i in range(1, N):                                            # Loops through interior nodes from 1 to n-1 since node 0 and N is known
    row = i - 1                                                  # Matrix row corresponding to node x_i

    b_vect[row] = (q * h**4 / (E * I))                           # RHS is same for all interior nodes, since q is constant throughout span

# First interior node : modified from using bending boundary conditions
# 5*v_1 - 4*v_2 + v_3 = q*h^4/(EI)
    if i == 1:
        A[row, row] =  5.0
        A[row, row + 1] = -4.0
        A[row, row + 2] =  1.0

# Second interior row: i = 2
    # -4v1 + 6v2 - 4v3 + v4 = q h^4 / (EI)
    elif i == 2:
        A[row, row - 1] = -4.0
        A[row, row] = 6.0
        A[row, row + 1] = -4.0
        A[row, row + 2] = 1.0

# Last interior node: modified from using bending boundary conditions
    # v_(N-3) - 4*v_(N-2) + 5*v_(N-1) = q*h^4/(EI)
    elif i == N - 1:
        A[row, row - 2] =  1.0
        A[row, row - 1] = -4.0
        A[row, row] =  5.0

 # Second last interior row: i = N-2
    # v_(N-4) - 4v_(N-3) + 6v_(N-2) - 4v_(N-1) = q h^4 / (EI)
    elif i == N - 2:
        A[row, row - 2] = 1.0
        A[row, row - 1] = -4.0
        A[row, row] = 6.0
        A[row, row + 1] = -4.0

    # Standard interior rows: five-point central difference method
    # v_(i-2) - 4*v_(i-1) + 6*v_i - 4*v_(i+1) + v_(i+2) = q*h^4/(EI)
    else:
        A[row, row - 2] = 1.0
        A[row, row - 1] = -4.0
        A[row, row] = 6.0
        A[row, row + 1] = -4.0
        A[row, row + 2] = 1.0


v_internal = np.linalg.solve(A, b_vect)                          # Solving for the interior nodal values

v[1:N] = v_internal                                              # Putting the interior solution back into the full array

In [93]:
# In this section I am doing any post-processing (if needed)

v_exact = q * (x**4 - 2.0 * L * x**3 + L**3 * x) / (24.0 * E * I) # Analytical solution
error = np.abs(v - v_exact)       # Absolute error
dvdx = np.gradient(v, h)          # Slope
d2vdx2 = np.gradient(dvdx, h)     # Curvature
M = -E * I * d2vdx2               # Bending moment (negative sign: v is positive downward)

# Maximum values
max_deflection = np.max(np.abs(v))# Max deflection magnitude
max_error = np.max(error)         # Max Error


# Building a rectangular beam mesh for visualisation of deformation:

ny = 20
y_width = np.linspace(-b / 2.0, b/ 2.0, ny)

# Mesh in the x-width plane
X_mesh, Y_mesh = np.meshgrid(x, y_width)

# Undeformed surface: z = 0
Z_undeformed = np.zeros_like(X_mesh)

# Deformed surface: beam deflection copied across the width
scale_factor = 50.0   # visual scaling so deformation can be seen clearly
Z_deformed = scale_factor * np.tile(v, (ny, 1))


print("Rectangular section width b =", b, "m")
print("Rectangular section depth d =", d, "m")
print("Second moment of area I0 =", I, "m^4")
print("Step size h =", h)
print("Maximum absolute error =", np.max(error))
print("Maximum deflection =", max_deflection)
# Elastic strain energy via Simpson's rule: U = integral_0^L M^2 / (2EI) dx
def Simpson_comp(xn, yn):
    h = xn[1] - xn[0]
    N = len(xn)
    S = 0
    for i in range(0, N-2, 2):
        S += h/3 * (yn[i] + 4*yn[i+1] + yn[i+2])
    return S

integrand = M**2 / (2.0 * E * I)
U_numerical = Simpson_comp(x, integrand)
U_analytical = (q**2 * L**5) / (240.0 * E * I)
U_error_pct = np.abs(U_numerical - U_analytical) / U_analytical * 100

print(f"Elastic strain energy (numerical, Simpson): {U_numerical:.3f} J")
print(f"Elastic strain energy (analytical):         {U_analytical:.3f} J")
print(f"Strain energy error = {U_error_pct:.2f}%")


Rectangular section width b = 0.08 m
Rectangular section depth d = 0.1 m
Second moment of area I0 = 6.666666666666668e-06 m^4
Step size h = 0.25
Maximum absolute error = 7.440476176344066e-06
Maximum deflection = 0.014888392857128722


In [ ]:
# Plotting the results

# Plot deflection
v_mm = v * 1e3
v_exact_mm = v_exact * 1e3

plt.figure(figsize=(10, 6))
plt.plot(x, v_mm, 'bo-', markersize=4, linewidth=1.8, label='Numerical Deflection')
plt.plot(x, v_exact_mm, 'r--', linewidth=2, label='Exact Deflection')
plt.xlabel('Position along beam, x (m)')
plt.ylabel('Deflection (mm)')
plt.title('Deflection of Simply Supported Beam under Uniformly Distributed Load')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Plot error
error_um = error * 1e6

plt.figure(figsize=(10, 6))
plt.plot(x, error_um, 'g-', linewidth=2, label='Absolute Error')
plt.xlabel('Position along beam, x (m)')
plt.ylabel('Error (μm)')
plt.title('Absolute Error between Numerical and Exact Solution')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Plot bending moment
M_kNm = M / 1e3

plt.figure(figsize=(10, 6))
plt.plot(x, M_kNm, 'm-', linewidth=2, label='Bending Moment')
plt.xlabel('Position along beam, x (m)')
plt.ylabel('Bending Moment (kN·m)')
plt.title('Bending Moment Distribution')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Plot shear force
V_num = np.gradient(M, x)
V_exact_sf = q * (L / 2.0 - x)
V_num_kN = V_num / 1e3
V_exact_sf_kN = V_exact_sf / 1e3

plt.figure(figsize=(10, 6))
plt.plot(x[1:-1], V_num_kN[1:-1], 'c-', linewidth=2, label='Numerical Shear Force')
plt.plot(x, V_exact_sf_kN, 'r--', linewidth=2, label='Exact Shear Force')
plt.axhline(0, color='k', linewidth=0.8, linestyle=':')
plt.xlabel('Position along beam, x (m)')
plt.ylabel('Shear Force (kN)')
plt.title('Shear Force Distribution')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 3D plot of deformed beam
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(X_mesh, Y_mesh, Z_deformed, color='blue', alpha=0.7)
ax.set_xlabel('Length (m)')
ax.set_ylabel('Width (m)')
ax.set_zlabel('Deflection (scaled)')
ax.set_title('3D Visualization of Beam Deformation')
plt.show()

# Contour plot of deflection over beam surface (length x width)
import numpy as np
V_contour = np.tile(v_mm, (ny, 1))  # deflection is uniform across width

plt.figure(figsize=(10, 4))
cp = plt.contourf(X_mesh, Y_mesh, V_contour, levels=20, cmap='viridis')
plt.colorbar(cp, label='Deflection (mm)')
plt.xlabel('Position along beam, x (m)')
plt.ylabel('Width (m)')
plt.title('Deflection Contour Plot')
plt.tight_layout()
plt.show()

In [95]:
# In this section I am celebrating
print('CW done: I deserve a good mark')

CW done: I deserve a good mark
